# 04 · Avaliação: placar das três configurações, alucinações e erros reais

Os mesmos **37 casos de teste** (`avaliacao/casos_teste.json`: 27 com resposta na base e 10 "não consta") são respondidos por:

| Configuração | O que é |
|---|---|
| **C1 · LLM base sem contexto** | `Qwen2.5-1.5B-Instruct` sozinho, sem a base (mesmo formato JSON pedido) |
| **C2 · RAG simples** | busca dos 3 trechos mais parecidos + modelo base |
| **C3 · RAG + LoRA** | mesma busca + modelo especializado no notebook 02 |
| **Sistema final** | C3 + limiar de recusa escolhido no notebook 01 (o que roda na interface e na API) |

C2 e C3 rodam **sem limiar**, para comparar só o efeito do LoRA. A geração é determinística (greedy) para o resultado ser reprodutível.

**Métricas** (calculadas por `src/agrofacil.py`, sem ajuste manual):
- `json_valido_%`: a saída é um JSON no formato `{resposta, fonte_id, status}`;
- `resposta_correta_%`: nas perguntas com resposta na base, o texto contém os termos do gabarito;
- `resposta_com_citacao_correta_%`: correta **e** citando o trecho certo;
- `recusa_correta_nao_consta_%`: nas perguntas fora da base, respondeu "não consta";
- `alucinacao_nao_consta_%`: nas perguntas fora da base, **inventou** uma resposta;
- `resposta_errada_afirmada_%`: nas perguntas da base, afirmou algo sem os termos do gabarito;
- `recusa_indevida_%`: recusou uma pergunta que tinha resposta.

In [ ]:
# Configuração do ambiente
# No Colab, o projeto fica no Google Drive para que todos os notebooks (01, 02, 04, 03)
# usem os mesmos arquivos: o limiar salvo no 01 e o adaptador LoRA treinado no 02.
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/palomari/agrofacil-genai.git"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive")
except ImportError:
    pass  # execução local

if not Path("src/agrofacil.py").exists():
    if Path("../src/agrofacil.py").exists():
        os.chdir("..")
    else:
        if not Path("agrofacil-genai").exists():
            subprocess.run(["git", "clone", REPO], check=True)
        os.chdir("agrofacil-genai")

assert Path("src/agrofacil.py").exists(), (
    "Versão antiga do repositório nesta pasta. Apague a pasta agrofacil-genai do Drive e rode de novo.")
sys.path.insert(0, os.getcwd())
print("Raiz do projeto:", os.getcwd())
print("Versão:", subprocess.run(["git", "log", "-1", "--format=%h %cd"], capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, re
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from src import agrofacil as af

testes = af.carregar_testes()
cfg = af.carregar_config()
print("Configuração da busca:", cfg)
buscador = af.Buscador(af.carregar_base(), cfg["encoder"])
model, tok = af.carregar_modelo(adapter=None)

PASTA = Path("avaliacao/resultados"); PASTA.mkdir(parents=True, exist_ok=True)
resultados = {}

def rodar(nome, **kw):
    res = [af.responder(c["pergunta"], model, tok, buscador=buscador, k=cfg.get("k", 3), **kw)
           for c in tqdm(testes, desc=nome)]
    slug = re.sub(r"[^a-z0-9]+", "_", af.normalizar(nome)).strip("_")
    with open(PASTA / f"{slug}.jsonl", "w", encoding="utf-8") as f:
        for c, r in zip(testes, res):
            f.write(json.dumps({"caso": c["id"], **r, **af.avaliar_caso(c, r)}, ensure_ascii=False) + "\n")
    resultados[nome] = res
    return res

In [ ]:
rodar("C1 · LLM base sem contexto", modo="sem_contexto")
rodar("C2 · RAG simples", modo="rag", limiar=None)

In [ ]:
assert af.CAMINHO_ADAPTER.exists(), "Rode o notebook 02 (ou baixe o adaptador para modelos/agrofacil_adapter)."
model = af.aplicar_adapter(model, af.CAMINHO_ADAPTER)
rodar("C3 · RAG + LoRA", modo="rag", limiar=None)
rodar("Sistema final · RAG + LoRA + limiar", modo="rag", limiar=cfg["limiar"])

## Placar

In [ ]:
placar = pd.DataFrame([af.resumo_config(n, testes, r) for n, r in resultados.items()])
placar.to_csv("avaliacao/placar.csv", index=False)
placar.set_index("configuracao").T

In [ ]:
import matplotlib.pyplot as plt

metricas = ["resposta_com_citacao_correta_%", "recusa_correta_nao_consta_%", "alucinacao_nao_consta_%", "json_valido_%"]
rotulos = ["Correta + citação", "Recusa correta", "Alucinação (fora da base)", "JSON válido"]
ax = placar.set_index("configuracao")[metricas].T.set_axis(rotulos).plot.bar(figsize=(10, 4), rot=0, width=.8)
ax.set_ylabel("%"); ax.set_ylim(0, 105); ax.legend(fontsize=8, loc="upper center", ncol=2)
ax.set_title("AgroFácil GenAI: placar das configurações (37 casos de teste)")
plt.tight_layout(); plt.savefig("avaliacao/placar.png", dpi=120); plt.show()

In [ ]:
# Interpretação automática a partir dos números (revise e complemente no README)
P = placar.set_index("configuracao")
c1, c2, c3, fin = P.index.tolist()
m = lambda cfgname, col: P.loc[cfgname, col]
linhas = [
    f"- **Base vs. RAG:** a resposta correta com citação vai de {m(c1,'resposta_com_citacao_correta_%')}% (C1, que não tem fonte para citar) "
    f"para {m(c2,'resposta_com_citacao_correta_%')}% no RAG simples; a resposta correta sem exigir citação vai de "
    f"{m(c1,'resposta_correta_%')}% para {m(c2,'resposta_correta_%')}%.",
    f"- **Alucinação fora da base:** C1 inventa resposta em {m(c1,'alucinacao_nao_consta_%')}% das perguntas 'não consta', "
    f"C2 em {m(c2,'alucinacao_nao_consta_%')}%, C3 em {m(c3,'alucinacao_nao_consta_%')}% e o sistema final em {m(fin,'alucinacao_nao_consta_%')}%.",
    f"- **Efeito do LoRA (C2 → C3):** JSON válido {m(c2,'json_valido_%')}% → {m(c3,'json_valido_%')}%; "
    f"correta + citação {m(c2,'resposta_com_citacao_correta_%')}% → {m(c3,'resposta_com_citacao_correta_%')}%; "
    f"recusa correta {m(c2,'recusa_correta_nao_consta_%')}% → {m(c3,'recusa_correta_nao_consta_%')}%; "
    f"recusa indevida {m(c2,'recusa_indevida_%')}% → {m(c3,'recusa_indevida_%')}%.",
    f"- **Efeito do limiar (C3 → final):** recusa correta {m(c3,'recusa_correta_nao_consta_%')}% → {m(fin,'recusa_correta_nao_consta_%')}%, "
    f"com recusa indevida {m(c3,'recusa_indevida_%')}% → {m(fin,'recusa_indevida_%')}% (o custo de recusar mais).",
    f"- **Latência média por pergunta (T4):** C1 {m(c1,'latencia_media_s')} s, C2 {m(c2,'latencia_media_s')} s, "
    f"C3 {m(c3,'latencia_media_s')} s, final {m(fin,'latencia_media_s')} s.",
]
interpretacao = "\n".join(linhas)
print(interpretacao)

In [ ]:
# Gera avaliacao/placar_comparativo.md a partir dos resultados (nada é digitado à mão)
def tabela_md(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join([" :--- "] + [" ---: "] * (len(cols) - 1)) + "|"]
    out += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(out)

import datetime
texto = f"""# Placar comparativo: AgroFácil GenAI

Gerado automaticamente por `notebooks/04_avaliacao.ipynb` em {datetime.date.today():%d/%m/%Y}.
Modelo: `{af.MODELO_BASE}` (4 bits NF4) · Encoder: `{cfg['encoder']}` · k = {cfg.get('k', 3)} · limiar = {cfg['limiar']}
Casos: {len(testes)} ({sum(c['tipo']=='esperado' for c in testes)} com resposta na base, {sum(c['tipo']=='nao_consta' for c in testes)} "não consta").

## Tabela

{tabela_md(placar)}

![placar](placar.png)

## Interpretação

{interpretacao}

## Arquivos

- Respostas completas de cada configuração: `avaliacao/resultados/*.jsonl`
- Detalhe por caso: `avaliacao/detalhe_por_caso.csv`
- Candidatos a erro com causa provável: `avaliacao/erros_candidatos.csv`
"""
Path("avaliacao/placar_comparativo.md").write_text(texto, encoding="utf-8")
print(texto)

## Detalhe por caso

In [ ]:
linhas = []
for c in testes:
    linha = {"caso": c["id"], "tipo": c["tipo"], "pergunta": c["pergunta"]}
    for n, res in resultados.items():
        r = res[c["id"] - 1]
        a = af.avaliar_caso(c, r)
        linha[n] = ("✔" if a["acerto"] else "✘") + f" {r['status']}" + (f" [{r['fonte_id']}]" if r["fonte_id"] else "")
    linhas.append(linha)
detalhe = pd.DataFrame(linhas)
detalhe.to_csv("avaliacao/detalhe_por_caso.csv", index=False)
pd.set_option("display.max_colwidth", 80)
detalhe

## Demonstração de alucinações

As mesmas perguntas **fora da base**, respondidas pelo LLM sem contexto (C1) e pelo sistema final. É a evidência pedida no relatório: o modelo sozinho inventa, e o sistema fundamentado recusa.

In [ ]:
c1n, finn = list(resultados)[0], list(resultados)[-1]
for c in testes:
    if c["tipo"] != "nao_consta":
        continue
    r1, rf = resultados[c1n][c["id"] - 1], resultados[finn][c["id"] - 1]
    print(f"[{c['id']}] {c['pergunta']}")
    print(f"   C1 (sem contexto) -> {r1['status']}: {r1['resposta'][:220]}")
    print(f"   Sistema final     -> {rf['status']} (origem: {rf['origem']}, score top-1 {rf['score_top1']:.3f})")
    print()

## Erros reais para o relatório (`avaliacao/5erros.md`)

A célula abaixo lista **todas** as falhas do C3 e do sistema final, com a causa provável detectada automaticamente (`recuperacao_errada`, `recusa_indevida_por_limiar`, `alucinacao_fora_da_base`, `json_invalido`, `citacao_errada`, ...). Escolham **5 erros de causas diferentes** e preencham `avaliacao/5erros.md` com o que o modelo respondeu, a causa e a correção proposta.

In [ ]:
erros = []
for n in list(resultados)[2:]:
    for c, r in zip(testes, resultados[n]):
        causa = af.diagnosticar(c, r)
        if causa != "ok":
            erros.append({"configuracao": n, "caso": c["id"], "tipo": c["tipo"], "pergunta": c["pergunta"],
                          "gabarito": c["gabarito"], "resposta_modelo": r["resposta"], "status": r["status"],
                          "fonte_citada": r["fonte_id"], "esperado": "/".join(c["doc_ids"]),
                          "top3": [t["id"] for t in r["trechos_recuperados"]], "score_top1": r["score_top1"],
                          "causa_provavel": causa})
df_erros = pd.DataFrame(erros)
df_erros.to_csv("avaliacao/erros_candidatos.csv", index=False)
print(f"{len(df_erros)} falhas registradas em avaliacao/erros_candidatos.csv")
if len(df_erros):
    print(df_erros.causa_provavel.value_counts())
    # sugestão: 1 erro de cada causa, priorizando o sistema final
    sugestao = (df_erros.sort_values("configuracao", ascending=False)
                .drop_duplicates("causa_provavel").head(5))
    display(sugestao[["configuracao", "caso", "pergunta", "resposta_modelo", "causa_provavel"]])